# Protocol 6 — Disorders of Consciousness Biomarker Model

Simulates predictions Pred 6.a–Pred 6.d from :

* **Pred 6.a** — HEP + PCI joint model explains more GCS-S variance than either biomarker alone
* **Pred 6.b** — HEP amplitude discriminates MCS from VS/UWS
* **Pred 6.c** — Interoceptive stimulation increases PCI in MCS but not VS/UWS
* **Pred 6.d** — HEP amplitude correlates with GCS-S at 3- and 6-month follow-up

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from apgi.core import compute_pi_i_eff

## 1 — Protocol 6 parameters

In [ ]:
# Estimated Pii by DoC group (from protocol_4 apgi_parameters)
PI_I_BY_GROUP = {"VS/UWS": 0.3, "MCS": 0.8, "Controls": 1.2}
N_PER_GROUP = {"VS/UWS": 15, "MCS": 20, "Controls": 15}
KAPPA = 100.0

rng = np.random.default_rng(7)
hep_by_group, pci_by_group = {}, {}
for group, pi_i in PI_I_BY_GROUP.items():
    n = N_PER_GROUP[group]
    C = rng.uniform(0.5, 2.0, n)
    pi_eff = compute_pi_i_eff(pi_i, C, kappa=KAPPA)
    hep_by_group[group] = pi_eff + rng.normal(0, 0.07, n)
    pci_base = {"VS/UWS": 0.18, "MCS": 0.35, "Controls": 0.52}[group]
    pci_by_group[group] = rng.normal(pci_base, 0.06, n)

for g in PI_I_BY_GROUP:
    print(f"{g:<12} HEP={hep_by_group[g].mean():.3f}  PCI={pci_by_group[g].mean():.3f}")

## 2 — Prediction 4.B: HEP discriminates MCS from VS/UWS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
groups = list(PI_I_BY_GROUP.keys())
colors = ["#d6604d", "#FFCC00", "#2166ac"]
x = range(len(groups))

ax = axes[0]
ax.bar(
    x,
    [hep_by_group[g].mean() for g in groups],
    yerr=[hep_by_group[g].std() / np.sqrt(N_PER_GROUP[g]) for g in groups],
    color=colors,
    alpha=0.85,
    edgecolor="white",
    capsize=5,
    width=0.4,
)
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylabel("HEP amplitude (a.u.)")
ax.set_title("Pred 6.b — HEP by DoC group")
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
ax.bar(
    x,
    [pci_by_group[g].mean() for g in groups],
    yerr=[pci_by_group[g].std() / np.sqrt(N_PER_GROUP[g]) for g in groups],
    color=colors,
    alpha=0.85,
    edgecolor="white",
    capsize=5,
    width=0.4,
)
ax.axhline(0.31, ls="--", lw=1, color="k", alpha=0.5, label="PCI threshold")
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylabel("PCI")
ax.set_title("PCI by DoC group")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Protocol 6 — Biomarkers across DoC groups", fontsize=12)
plt.tight_layout()
plt.show()

## 3 — Prediction 4.A: Joint model R² vs. univariate models

In [ ]:
all_hep, all_pci, all_gcs = [], [], []
for group in groups:
    n = N_PER_GROUP[group]
    base_gcs = {"VS/UWS": 4.0, "MCS": 9.0, "Controls": 14.5}[group]
    gcs = (
        base_gcs
        + 2.0 * hep_by_group[group]
        + 3.0 * pci_by_group[group]
        + rng.normal(0, 1.2, n)
    )
    all_hep.extend(hep_by_group[group])
    all_pci.extend(pci_by_group[group])
    all_gcs.extend(gcs)
all_hep = np.array(all_hep)
all_pci = np.array(all_pci)
all_gcs = np.array(all_gcs)


def r2(y, *Xs):
    X = np.column_stack([np.ones(len(y))] + list(Xs))
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    ss_res = np.sum((y - X @ b) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot


r2_hep = r2(all_gcs, all_hep)
r2_pci = r2(all_gcs, all_pci)
r2_joint = r2(all_gcs, all_hep, all_pci)
print(f"R² HEP only:    {r2_hep:.3f}")
print(f"R² PCI only:    {r2_pci:.3f}")
print(f"R² Joint model: {r2_joint:.3f}  (Pred 6.a: joint > max univariate)")

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ["HEP only", "PCI only", "Joint"],
    [r2_hep, r2_pci, r2_joint],
    color=["#2166ac", "#9966FF", "#00CC99"],
    alpha=0.85,
    edgecolor="white",
    width=0.4,
)
for bar, v in zip(bars, [r2_hep, r2_pci, r2_joint]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        v + 0.01,
        f"{v:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylabel("R² for GCS-S recovery")
ax.set_title("Pred 6.a — Joint model superiority")
ax.set_ylim(0, 1)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()